# SO-101 Tic-Tac-Toe — Games 1–15 SmolVLA Baseline (A100)

Bu notebook fiziksel SO-101 ile kaydedilmiş ilk **15 oyun / 195 episode** datasetinden erken bir
SmolVLA baseline üretir. Hub'daki dataset revision'i run basinda SHA'ya sabitlenir ve 195 episode degilse egitim baslamaz. Dataset şeması
`observation.images.top` + `observation.images.wrist`, 6D state/action ve 30 FPS olarak kilitlenir.

Token hiçbir hücreye yazılmaz; Colab Secrets içindeki `HF_TOKEN` kullanılır. Runtime başlamadan
**A100** seçilmelidir.

Çıktı model: `HashtagRobotics/smolvla-tic-tac-toe-games-1-15-120k`


## 1. LeRobot 0.6.1 ve SmolVLA bağımlılıklarını kur


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "lerobot[dataset,smolvla,training]==0.6.1",
    "huggingface_hub>=0.34",
    "wandb",
])


## 2. HF/opsiyonel W&B auth ve zorunlu A100 kontrolü


In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import HfApi, login


def get_secret(name: str):
    try:
        return userdata.get(name)
    except Exception:
        return None


HF_TOKEN = (get_secret("HF_TOKEN") or "").strip()
if not HF_TOKEN:
    raise RuntimeError(
        "Colab > Secrets alanina HF_TOKEN ekle ve notebook erisimini ac. Tokeni hucreye yapistirma."
    )
login(token=HF_TOKEN, add_to_git_credential=False)
identity = HfApi(token=HF_TOKEN).whoami()
print("Hugging Face user:", identity["name"])

if not torch.cuda.is_available():
    raise RuntimeError("CUDA yok. Colab Runtime > Change runtime type > A100 GPU sec.")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {GPU_NAME} ({GPU_GB:.1f} GiB)")
if "A100" not in GPU_NAME.upper():
    raise RuntimeError("Bu notebook A100 profiline kilitli; secilen GPU A100 degil.")
torch.set_float32_matmul_precision("high")

WANDB_API_KEY = get_secret("WANDB_API_KEY")
USE_WANDB = bool(WANDB_API_KEY)
if USE_WANDB:
    import wandb

    wandb.login(key=WANDB_API_KEY, relogin=True)
print("Weights & Biases:", "enabled" if USE_WANDB else "disabled (optional secret missing)")


## 3. Dataset, model ve tekrar üretilebilir deney kimlikleri


In [ ]:
import json
from datetime import UTC, datetime
from pathlib import Path

from huggingface_hub import HfApi, snapshot_download

DATASET_REPO = "HashtagRobotics/tic-tac-toe-so101-block-a-clean-v1"
DATASET_REVISION = "b1a5e8681619bd5352c29f0261843828503f1643"
EXPECTED_EPISODES = 195
EXPECTED_CAMERAS = ["observation.images.top", "observation.images.wrist"]
EXPECTED_TASKS = 18
MODEL_REPO = "HashtagRobotics/smolvla-tic-tac-toe-games-1-15-120k"
BASE_MODEL = "lerobot/smolvla_base"
BASE_REVISION = "c83c3163b8ca9b7e67c509fffd9121e66cb96205"
RENAME_MAP = {
    "observation.images.top": "observation.images.camera1",
    "observation.images.wrist": "observation.images.camera2",
}
EMPTY_CAMERAS = 1
BATCH_SIZE = 16
STEPS = 120_000
CHECKPOINT_FREQ = 20_000
EVAL_FREQ = 20_000
SEED = 42

api = HfApi(token=HF_TOKEN)
dataset_info = api.dataset_info(DATASET_REPO, revision=DATASET_REVISION)
assert dataset_info.sha == DATASET_REVISION
print("Dataset:", DATASET_REPO)
print("Dataset revision:", DATASET_REVISION)
print("Base revision:", BASE_REVISION)


## 4. Hub dataset bütünlük ve iki-kamera schema kapısı


In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

DEFAULT_EXPECTED_CAMERAS = ["observation.images.front", "observation.images.wrist"]


def verify_dataset_meta(
    repo_id: str,
    revision: str,
    expected_episodes: int,
    expected_cameras=DEFAULT_EXPECTED_CAMERAS,
    expected_tasks: int = 18,
):
    meta = LeRobotDatasetMetadata(repo_id=repo_id, revision=revision, token=HF_TOKEN)
    camera_keys = sorted(meta.video_keys)
    state_shape = list(meta.features["observation.state"]["shape"])
    action_shape = list(meta.features["action"]["shape"])
    if hasattr(meta.episodes, "items"):
        episode_rows = [
            {"episode_index": index, **episode}
            for index, episode in meta.episodes.items()
        ]
    else:
        episode_rows = [meta.episodes[index] for index in range(len(meta.episodes))]
    episode_indices = sorted(int(episode["episode_index"]) for episode in episode_rows)
    zero_frame_episodes = [
        int(episode["episode_index"])
        for episode in episode_rows
        if int(episode.get("length", 0)) <= 0
    ]
    checks = {
        "episodes": meta.total_episodes,
        "frames": meta.total_frames,
        "fps": meta.fps,
        "robot_type": meta.robot_type,
        "camera_keys": camera_keys,
        "state_shape": state_shape,
        "action_shape": action_shape,
        "tasks": len(meta.tasks),
        "zero_frame_episodes": zero_frame_episodes,
    }
    assert meta.total_episodes == expected_episodes, checks
    assert episode_indices == list(range(expected_episodes)), checks
    assert meta.fps == 30, checks
    assert camera_keys == sorted(expected_cameras), checks
    assert state_shape == [6], checks
    assert action_shape == [6], checks
    assert meta.total_frames > 0, checks
    assert len(meta.tasks) == expected_tasks, checks
    assert not zero_frame_episodes, checks
    print(json.dumps(checks, indent=2, ensure_ascii=False))
    return meta, checks


In [ ]:
meta, dataset_report = verify_dataset_meta(
    DATASET_REPO,
    DATASET_REVISION,
    EXPECTED_EPISODES,
    expected_cameras=EXPECTED_CAMERAS,
    expected_tasks=EXPECTED_TASKS,
)


## 5. SmolVLA base checkpoint'ini sabit revision ile indir


In [ ]:
BASE_PATH = Path("/content/models/smolvla_base_c83c316")
snapshot_download(
    repo_id=BASE_MODEL,
    revision=BASE_REVISION,
    local_dir=BASE_PATH,
    token=HF_TOKEN,
)
assert (BASE_PATH / "model.safetensors").is_file()
print("Pinned base:", BASE_PATH)


## 6. Eğitimi başlat

Bu yayımlanmış run A100 40 GB için batch 16 / 120k step full fine-tune profilini kullanır.
SmolVLA'nın BF16 ağırlıkları ile FP16 GradScaler çakışmasını önlemek için `use_amp=false` sabittir.
Episode'ların %10'u offline
eval için ayrılır. Her 20k stepte checkpoint ve eval üretilip Hugging Face'e yüklenir; 20k/40k/60k/80k/100k/120k sonuçlarını karşılaştır,
son checkpoint'in otomatik olarak en iyi olduğunu varsayma. `resume=false` kalmalıdır.


In [ ]:
import subprocess
from collections import deque

RENAME_MAP = {
    "observation.images.top": "observation.images.camera1",
    "observation.images.wrist": "observation.images.camera2",
}
EMPTY_CAMERAS = 1

def train_command(*, dataset_repo, dataset_revision, model_repo, base_path, output_dir, steps):
    return [
        "lerobot-train",
        f"--policy.path={base_path}",
        f"--dataset.repo_id={dataset_repo}",
        f"--dataset.revision={dataset_revision}",
        "--dataset.eval_split=0.1",
        "--dataset.return_uint8=true",
        f"--policy.empty_cameras={EMPTY_CAMERAS}",
        f"--rename_map={json.dumps(RENAME_MAP, separators=(',', ':'))}",
        f"--policy.repo_id={model_repo}",
        "--policy.device=cuda",
        "--policy.use_amp=false",
        "--policy.freeze_vision_encoder=false",
        "--policy.train_expert_only=false",
        "--policy.optimizer_lr=0.0001",
        f"--policy.scheduler_decay_steps={steps}",
        f"--batch_size={BATCH_SIZE}",
        f"--steps={steps}",
        f"--save_freq={CHECKPOINT_FREQ}",
        f"--eval_steps={EVAL_FREQ}",
        "--max_eval_samples=2048",
        "--num_workers=4",
        "--seed=42",
        "--resume=false",
        f"--output_dir={output_dir}",
        f"--job_name={model_repo.rsplit('/', 1)[-1]}",
        f"--wandb.enable={str(USE_WANDB).lower()}",
        "--policy.push_to_hub=true",
        "--save_checkpoint_to_hub=true",
    ]

RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = Path(f"/content/outputs/smolvla_tic_tac_toe_games_1_15_120k_{RUN_ID}")
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
launch = {
    "stage": "GAMES_1_15_BASELINE",
    "dataset_repo": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "dataset_report": dataset_report,
    "base_model": BASE_MODEL,
    "base_revision": BASE_REVISION,
    "model_repo": MODEL_REPO,
    "rename_map": RENAME_MAP,
    "empty_cameras": EMPTY_CAMERAS,
    "batch_size": BATCH_SIZE,
    "use_amp": False,
    "steps": STEPS,
    "checkpoint_freq": CHECKPOINT_FREQ,
    "eval_freq": EVAL_FREQ,
    "seed": SEED,
    "eval_split": 0.1,
    "resume": False,
}
Path("/content/games_1_15_launch_manifest.json").write_text(
    json.dumps(launch, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
command = train_command(
    dataset_repo=DATASET_REPO,
    dataset_revision=DATASET_REVISION,
    model_repo=MODEL_REPO,
    base_path=BASE_PATH,
    output_dir=OUTPUT_DIR,
    steps=STEPS,
)
command.append("--policy.private=true")
assert "--resume=false" in command
print("Launching:", " ".join(map(str, command)))
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tail = deque(maxlen=120)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
    tail.append(line)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f"lerobot-train exit code {return_code}. Son loglar:\n{''.join(tail)}"
    )

model_info = api.model_info(MODEL_REPO)
print("Model:", MODEL_REPO)
print("Commit SHA:", model_info.sha)
print("Training tamamlandı. Fiziksel rollout öncesi checkpoint schema/load-only/E-STOP preflight zorunlu.")
